# Inductive bias: trees vs neural networks

Playground on two 2D binary classification tasks:
- **Smooth**: sine boundary
- **Square**: step boundary

We will:
1. Train a decision tree and a smooth neural network for both tasks.
2. See how each model's inductive bias fits a smooth vs a sharp boundary.
3. Play with the NN hyperparameters to watch its inductive bias shift.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from ipywidgets import interact_manual, Dropdown, IntSlider, FloatLogSlider, IntText
from IPython.display import display
import pandas as pd


np.random.seed(0)
torch.manual_seed(0)

## Data
Build one train/test set per boundary.

In [ ]:
def smooth_boundary(x1):
    return 0.75 * np.sin(2.5 * x1)

def square_boundary(x1):
    return np.where(np.sin(3.0 * x1) >= 0.0, 0.65, -0.65)

def make_dataset(boundary_fn, n=400, seed=0):
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(-1.8, 1.8, size=n)
    x2 = rng.uniform(-1.5, 1.5, size=n)
    X = np.column_stack([x1, x2])
    y = (x2 > boundary_fn(x1)).astype(int)
    return X, y

def smooth_diagnostic():
    # pairs of points just above/below the sine boundary
    x1s = np.linspace(-1.5, 1.5, 8)
    offset = 0.1
    pts = [[x1, smooth_boundary(x1) + s * offset] for x1 in x1s for s in (-1, 1)]
    X = np.array(pts)
    y = (X[:, 1] > smooth_boundary(X[:, 0])).astype(int)
    return X, y

def square_diagnostic():
    # points straddling the square-wave jump discontinuities
    jumps = np.array([-np.pi / 3, 0.0, np.pi / 3])
    offset = 0.05
    x2s = [-0.55, 0.0, 0.55]
    pts = [[j + s, x2] for j in jumps for s in (-offset, offset) for x2 in x2s]
    X = np.array(pts)
    y = (X[:, 1] > square_boundary(X[:, 0])).astype(int)
    return X, y

N_train = 400
N_test = 400

datasets = {
    "smooth": {
        "boundary_fn": smooth_boundary,
        "train": make_dataset(smooth_boundary, n=N_train, seed=1),
        "test":  make_dataset(smooth_boundary, n=N_test, seed=2),
        "diag":  smooth_diagnostic(),
    },
    "square": {
        "boundary_fn": square_boundary,
        "train": make_dataset(square_boundary, n=N_train, seed=3),
        "test":  make_dataset(square_boundary, n=N_test, seed=4),
        "diag":  square_diagnostic(),
    },
}

## Models

In [ ]:
def train_tree(X, y, max_depth=8, min_samples_leaf=4):
    tree = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=0,
    )
    tree.fit(X, y)
    return tree

ACTIVATIONS = {"tanh": nn.Tanh, "relu": nn.ReLU, "gelu": nn.GELU, "sigmoid": nn.Sigmoid}

class MLP(nn.Module):
    def __init__(self, hidden_dim=8, n_hidden_layers=2, activation="tanh"):
        super().__init__()
        act = ACTIVATIONS[activation.lower()]
        layers = [nn.Linear(2, hidden_dim), act()]
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), act()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def train_mlp(
    X, y,
    hidden_dim=8,
    n_hidden_layers=2,
    activation="tanh",
    epochs=2000,
    lr=1e-2,
    weight_decay=2.5e-3,
    loss="bce",
):
    torch.manual_seed(0)
    model = MLP(hidden_dim, n_hidden_layers, activation)
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y[:, None], dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    # bce = log-likelihood on probabilities; mse = regress directly to {0,1}
    loss_fn = nn.BCEWithLogitsLoss() if loss == "bce" else nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad()
        out = model(Xt)
        pred = out if loss == "bce" else torch.sigmoid(out)
        loss_fn(pred, yt).backward()
        opt.step()
    return model

@torch.no_grad()
def predict_mlp(model, X):
    return torch.sigmoid(model(torch.tensor(X, dtype=torch.float32))).numpy().ravel()

def predict_tree(tree, X):
    return tree.predict_proba(X)[:, 1]

## Plotting

In [ ]:
# set up grid properties
grid_size = 250
x1_range = 1.8
x2_range = 1.5
x1_grid = np.linspace(-x1_range, x1_range, grid_size)
x2_grid = np.linspace(-x2_range, x2_range, grid_size)
xx, yy = np.meshgrid(x1_grid, x2_grid)
X_grid = np.column_stack([xx.ravel(), yy.ravel()])

def draw_boundary(ax, boundary_fn, kind):
    x = np.linspace(-x1_range, x1_range, 1000)
    y = boundary_fn(x)
    style = dict(color="black", linestyle="--", linewidth=2)
    if kind == "square":
        ax.step(x, y, where="post", **style)
    else:
        ax.plot(x, y, **style)

def overlay_diag(ax, X_diag, y_diag, pred=None):
    # circles = correct, crosses = wrong (when pred given); else just show labels
    if pred is None:
        ax.scatter(X_diag[:, 0], X_diag[:, 1], c=y_diag, cmap="coolwarm",
                   s=70, edgecolors="black", linewidths=1.3)
    else:
        ok = pred == y_diag
        ax.scatter(X_diag[ok, 0], X_diag[ok, 1], marker="o", s=80,
                   facecolors="none", edgecolors="black", linewidths=2)
        ax.scatter(X_diag[~ok, 0], X_diag[~ok, 1], marker="x", s=80,
                   color="black", linewidths=2)

def plot_surface(ax, probs, boundary_fn, kind, X_scatter, y_scatter, title,
                 scatter_pred=None, X_diag=None, y_diag=None, diag_pred=None):
    ax.contourf(xx, yy, probs.reshape(xx.shape), levels=30, cmap="coolwarm", vmin=0, vmax=1)
    ax.contour(xx, yy, probs.reshape(xx.shape), levels=[0.5], colors="black", linewidths=2)
    draw_boundary(ax, boundary_fn, kind)
    if scatter_pred is None:
        ax.scatter(X_scatter[:, 0], X_scatter[:, 1], c=y_scatter, s=15, cmap="coolwarm",
                   alpha=0.5, edgecolor="white", linewidth=1)
    else:
        ok = scatter_pred == y_scatter
        ax.scatter(X_scatter[ok, 0], X_scatter[ok, 1], c=y_scatter[ok], s=15, cmap="coolwarm",
                   alpha=0.5, edgecolor="white", linewidth=1)
        ax.scatter(X_scatter[~ok, 0], X_scatter[~ok, 1], marker="x", s=30,
                   color="white", linewidths=0.5)
    if X_diag is not None:
        overlay_diag(ax, X_diag, y_diag, diag_pred)
    ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$")
    ax.set_title(title)

## Ground truth
Training data, true boundary, and some diagnostic test points near the decision boundary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

for col, kind in enumerate(["smooth", "square"]):
    d = datasets[kind]
    X_tr, y_tr = d["train"]
    X_dg, y_dg = d["diag"]
    bfn = d["boundary_fn"]
    ax = axes[col]

    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, s=12, cmap="coolwarm",
               alpha=0.7, edgecolor="none", label="training data")
    draw_boundary(ax, bfn, kind)
    overlay_diag(ax, X_dg, y_dg)
    ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel(r"$x_1$"); ax.set_ylabel(r"$x_2$")
    ax.set_title(f"{kind} boundary")
plt.show()

## (A) Tree vs NN

Comparing a simple NN (small MLP) to a decision tree for a task with smooth vs sharp decision boundaries.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)

for row, kind in enumerate(["smooth", "square"]):
    d = datasets[kind]
    X_tr, y_tr = d["train"]
    X_te, y_te = d["test"]
    X_dg, y_dg = d["diag"]
    bfn = d["boundary_fn"]

    tree = train_tree(X_tr, y_tr)
    mlp = train_mlp(X_tr, y_tr)
    
    tree_te_pred =  tree.predict(X_te)
    mlp_te_pred = predict_mlp(mlp, X_te) > 0.5
    
    tree_te = accuracy_score(y_te, tree_te_pred)
    mlp_te = accuracy_score(y_te, mlp_te_pred)
    
    
    tree_dg_pred = tree.predict(X_dg)
    mlp_dg_pred = (predict_mlp(mlp, X_dg) > 0.5).astype(int)
    tree_dg = accuracy_score(y_dg, tree_dg_pred)
    mlp_dg = accuracy_score(y_dg, mlp_dg_pred)

    plot_surface(axes[row, 0], predict_tree(tree, X_grid), bfn, kind, X_te, y_te, 
                 f"{kind} | Tree (test={tree_te:.3f}, diag={tree_dg:.3f})",
                 tree_te_pred, X_dg, y_dg, tree_dg_pred)
    plot_surface(axes[row, 1], predict_mlp(mlp, X_grid), bfn, kind, X_te, y_te,
                 f"{kind} | NN (test={mlp_te:.3f}, diag={mlp_dg:.3f})",
                 mlp_te_pred, X_dg, y_dg, mlp_dg_pred)

plt.show()

## (B) NN hyperparameter playground
Play with the widget to retrain the NN on both datasets so you can see how the inductive bias shifts.

Knobs:
- `activation`: `tanh`, `relu`, `gelu`, `sigmoid`
- `hidden_dim`, `n_hidden_layers`: capacity
- `weight_decay`: L2 regularization strength
- `loss`: `bce` or `mse`

If you want to play with it, try to find hyperparameters that fit both sets of diagnostic points - of course this is not as easy in reality when we can't see the true decision boundaries ;)

In [ ]:
logbook = []

def run(activation, hidden_dim, n_hidden_layers, weight_decay, loss, epochs, lr):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    run_scores = {}

    for col, kind in enumerate(["smooth", "square"]):
        d = datasets[kind]
        X_tr, y_tr = d["train"]
        X_te, y_te = d["test"]
        X_dg, y_dg = d["diag"]
        bfn = d["boundary_fn"]

        mlp = train_mlp(X_tr, y_tr,
                        activation=activation,
                        hidden_dim=hidden_dim,
                        n_hidden_layers=n_hidden_layers,
                        weight_decay=weight_decay,
                        loss=loss,
                        epochs=epochs,
                        lr=lr)
        test_pred = predict_mlp(mlp, X_te) > 0.5
        test_acc = accuracy_score(y_te, test_pred)
        dg_pred = (predict_mlp(mlp, X_dg) > 0.5).astype(int)
        dg_acc = accuracy_score(y_dg, dg_pred)
        
        run_scores[kind] = (test_acc, dg_acc)
        
        plot_surface(axes[col], predict_mlp(mlp, X_grid), bfn, kind, X_te, y_te,
                     f"{kind} | test={test_acc:.3f}, diag={dg_acc:.3f}",
                     test_pred, X_dg, y_dg, dg_pred)
    fig.suptitle(f"{activation} | dim={hidden_dim} | layers={n_hidden_layers} "
                 f"| wd={weight_decay:.1e} | loss={loss}", y=1.05)
    plt.show()
    
    logbook.append({
        "act": activation,
        "dim": hidden_dim,
        "layers": n_hidden_layers,
        "wd": f"{weight_decay:.1e}",
        "loss": loss,
        "epochs": epochs,
        "lr": f"{lr:.1e}",
        "smooth_test": f"{run_scores['smooth'][0]:.3f}",
        "smooth_diag": f"{run_scores['smooth'][1]:.3f}",
        "square_test": f"{run_scores['square'][0]:.3f}",
        "square_diag": f"{run_scores['square'][1]:.3f}",
    })

    display(pd.DataFrame(logbook))

interact_manual(
    run,
    activation=Dropdown(options=["tanh", "relu", "gelu", "sigmoid"], value="tanh"),
    hidden_dim=IntSlider(min=2, max=64, step=1, value=8),
    n_hidden_layers=IntSlider(min=1, max=6, step=1, value=2),
    weight_decay=FloatLogSlider(base=10, min=-6, max=0, step=0.5, value=2.5e-3),
    loss=Dropdown(options=["bce", "mse"], value="bce"),
    epochs=IntText(value=2000),
    lr=FloatLogSlider(base=10, min=-4, max=-1, step=0.5, value=1e-2),
);